# **Detecção de Fraudes em Pagamentos Online**

**Objetivo:**
* Encontrar o melhor modelo de machine learning e treiná-lo para ser capaz de identificar possíveis fraudes em pagamentos online.

### **1. Introdução**

<p align="justify">
Nos últimos anos, os pagamentos online revolucionaram a forma como realizamos transações financeiras. A comodidade e a agilidade associadas a esse método de pagamento o tornaram uma escolha preferencial para consumidores e empresas em todo o mundo. No entanto, com o aumento na popularidade dos pagamentos online, surgiram desafios significativos relacionados à segurança, em especial a ocorrência de fraudes em pagamentos online.

<p align="justify">
A detecção de fraudes em pagamentos online é um tópico de extrema importância e relevância no cenário financeiro atual. À medida que os métodos de fraude se tornam mais sofisticados, é fundamental que as empresas e instituições financeiras desenvolvam abordagens igualmente avançadas para detectar e prevenir essas atividades fraudulentas.

<p align="justify">
Ao longo deste projeto, exploraremos as diversas facetas da detecção de fraudes em pagamentos online, desde a coleta e preparação de dados até a implementação de modelos de Machine Learning e a validação de seu desempenho.

In [ ]:
# 1.1 Instalando biblioteca

!pip install pycaret

In [ ]:
# 1.2 Carregando bibliotecas

import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt

from pycaret.classification import *
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [ ]:
# 1.3 Carregando base de dados

dados = pd.read_csv('/content/drive/MyDrive/Portfólio/Dados/Detecção de Fraudes.csv',
                    sep = ',')

In [ ]:
# 1.4 Visualizando base de dados

dados.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


## **2. Análise Exploratória de Dados**

In [ ]:
# 2.1 Verificando shape dos dados

dados.shape

(6362620, 11)

In [ ]:
# 2.2 Verificando a existência de valores faltantes

dados.isnull().sum()

,0
step,0
type,0
amount,0
nameOrig,0
oldbalanceOrg,0
newbalanceOrig,0
nameDest,0
oldbalanceDest,0
newbalanceDest,0
isFraud,0


In [ ]:
# 2.3 Verificando informações sobre o dataset

dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


In [ ]:
# 2.4 Verificando o tipo e total de cada transação

dados.type.value_counts()

,count
type,
CASH_OUT,2237500
PAYMENT,2151495
CASH_IN,1399284
TRANSFER,532909
DEBIT,41432


In [ ]:
# 2.5 Verificando o total de informações por tipo: 0(No Fraud) - 1(Fraud)

dados.isFraud.value_counts()

,count
isFraud,
0,6354407
1,8213


In [ ]:
# 2.6 Gráfico 1: distribuição das transações

## 2.6.1 Selecionando informações
tipo = dados['type'].value_counts()
transacoes = tipo.index
quantidade = tipo.values

## 2.6.2 Plotando gráfico
figure = px.pie(dados,
                values = quantidade,
                names = transacoes,
                hole = 0.5)

figure.update_layout(title = 'Distribuição das Transações',
                     title_x = 0.5)
figure.show();

## **3. Pré-Processamento de Dados**

In [ ]:
# 3.1 Realizando transformações na coluna type: de categórico para númerico

dados['type'] = dados['type'].map({'CASH_OUT': 1, 'PAYMENT': 2,
                                   'CASH_IN': 3, 'TRANSFER': 4,
                                   'DEBIT': 5})

In [ ]:
# 3.2 Verificando transformações

dados['type'].unique()

array([2, 4, 1, 5, 3])

In [ ]:
# 3.3 Realizando transformação na coluna isFraud: de númerico para categórico

dados['isFraud'] = dados['isFraud'].map({0: 'No Fraud',
                                         1: 'Fraud'})

In [ ]:
# 3.4 Veificando transformações

dados['isFraud'].unique()

array(['No Fraud', 'Fraud'], dtype=object)

In [ ]:
# 3.5 Separando os dados em previsores e classes

## 3.5.1 Previsores
previsores = np.array(dados[["type", "amount", "oldbalanceOrg",
                             "newbalanceOrig", "oldbalanceDest",
                             "newbalanceDest"]])
## 3.5.2 Classes
classes = dados['isFraud']

In [ ]:
# 3.6 Dividindo os dados em treino e teste

## 70% para treino e 30% para teste
X_treino, X_teste, y_treino, y_teste = train_test_split(previsores,
                                                        classes,
                                                        test_size = 0.3,
                                                        random_state = 0)

In [ ]:
# 3.7 Verificando como ficou a distribuição dos dados após a divisão

print(X_treino.shape)
print(y_treino.shape)
print(X_teste.shape)
print(y_teste.shape)

(4453834, 6)
(4453834,)
(1908786, 6)
(1908786,)


## **4. Construindo Modelo**

**Observação**

<p align="justify">
O PyCaret ajuda a encontrar o melhor modelo para os dados a partir da variável target informada. Entretanto, por se ter um conjunto de dados muito grande o procedimento será demorado.

<p align="justify">
Para evitar essa demora, será selecionada uma amostra dos dados a partir da técnica de Amostragem Aleatória Simples. Será selecionado 10% de todo o conjunto de dados.

In [ ]:
# 4.1 Selecionando amostra do conjunto de dados

## 4.1.1 frac = 0.10 -> refere-se a 10%
amostra = dados.sample(frac = 0.10)

In [ ]:
# 4.2 Tamanho da amostra selecionada

## 4.2.1 -> 636.262 linhas e 11 colunas
amostra.shape

(636262, 11)

In [ ]:
# 4.3 Verificando melhor técnica a se usar com o PyCaret

clf = setup(data = amostra,
            target = 'isFraud',
            session_id = 0)

compare_models()

,Description,Value
0,Session id,0
1,Target,isFraud
2,Target type,Binary
3,Target mapping,"Fraud: 0, No Fraud: 1"
4,Original data shape,"(636262, 11)"
5,Transformed data shape,"(636262, 11)"
6,Transformed train set shape,"(445383, 11)"
7,Transformed test set shape,"(190879, 11)"
8,Numeric features,8
9,Categorical features,2


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
knn,K Neighbors Classifier,0.9994,0.8785,0.9994,0.9993,0.9993,0.7263,0.7382,17.9820
lr,Logistic Regression,0.9992,0.9465,0.9992,0.9992,0.9991,0.6298,0.6601,14.3370
dt,Decision Tree Classifier,0.9987,0.5000,0.9987,0.9973,0.9980,0.0000,0.0000,9.4150
ridge,Ridge Classifier,0.9987,0.9430,0.9987,0.9973,0.9980,0.0000,0.0000,9.0140
rf,Random Forest Classifier,0.9987,0.7001,0.9987,0.9973,0.9980,0.0000,0.0000,33.4990
ada,Ada Boost Classifier,0.9987,0.5000,0.9987,0.9973,0.9980,0.0000,0.0000,9.2170
gbc,Gradient Boosting Classifier,0.9987,0.4963,0.9987,0.9973,0.9980,0.0000,0.0000,36.7660
lda,Linear Discriminant Analysis,0.9987,0.3406,0.9987,0.9973,0.9980,0.0000,0.0000,9.7000
et,Extra Trees Classifier,0.9987,0.6967,0.9987,0.9973,0.9980,0.0000,0.0000,14.3190
lightgbm,Light Gradient Boosting Machine,0.9987,0.5000,0.9987,0.9973,0.9980,0.0000,0.0000,12.7770


Processing:   0%|          | 0/65 [00:00<?, ?it/s]

KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
                     metric_params=None, n_jobs=-1, n_neighbors=5, p=2,
                     weights='uniform')

**Resultado:** a partir dos comparativos realizados pelo PyCaret, o melhor modelo para se usar nos dados é o KNN.

In [ ]:
# 4.4 Carregando biblioteca

from sklearn.neighbors import KNeighborsClassifier

In [ ]:
# 4.4 Instânciando Modelo

modelo = KNeighborsClassifier(algorithm = 'auto',
                              leaf_size = 30,
                              metric = 'minkowski',
                              metric_params = None,
                              n_jobs = -1,
                              n_neighbors = 5,
                              p = 2,
                              weights = 'uniform')

In [ ]:
# 4.6 Treinando modelo

modelo.fit(X_teste, y_teste)

KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
                     metric_params=None, n_jobs=-1, n_neighbors=5, p=2,
                     weights='uniform')

In [ ]:
# 4.7 Aplicando modelo aos dados de teste

y_predito = modelo.predict(X_teste)

## **5. Avaliando Modelo**

In [ ]:
# 5.1 Métricas de Avaliação

print(classification_report(y_teste, y_predito))

              precision    recall  f1-score   support

       Fraud       0.90      0.68      0.78      2419
    No Fraud       1.00      1.00      1.00   1906367

    accuracy                           1.00   1908786
   macro avg       0.95      0.84      0.89   1908786
weighted avg       1.00      1.00      1.00   1908786



## **6. Pré-Processamento dos Dados**

<p align="justify">
Após os resultados obtidos, em especial na classe Fraud, aplicaremos mais algumas etapas de pré-processamento de dados com o objetivo de melhorar o desempenho de classificação da respectiva classe, em especial o indices recall e f1-score.

<p align="justify">
A primeira etapa a ser aplicada será o Balanceamento de Classes, tanto para aumentar a classe minoritária como também para diminuir a majoritária. Após isso, treinaremos o modelo novamente e analizaremos os resultados.

<p align="justify">
É possível que também possamos realizar o escalonamento/normalização dos dados.

In [ ]:
# 6.1 Carregando bibliotecas

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [ ]:
# 6.2 Oversampling: aumento da classe minoritária

smote = SMOTE(sampling_strategy = 'auto', random_state = 0)
X_treino_over, y_treino_over = smote.fit_resample(X_treino, y_treino)
print(X_treino_over.shape)
print(y_treino_over.shape)

(8896080, 6)
(8896080,)


In [ ]:
# 6.3 Treinando modelo com os dados após Oversample

## 6.3.1 Instânciando modelo
modelo_over = KNeighborsClassifier(algorithm = 'auto',
                                   leaf_size = 30,
                                   metric = 'minkowski',
                                   metric_params = None,
                                   n_jobs = -1,
                                   n_neighbors = 5,
                                   p = 2,
                                   weights = 'uniform')

## 6.3.2 Treinando modelo
modelo_over.fit(X_treino_over, y_treino_over)

KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
                     metric_params=None, n_jobs=-1, n_neighbors=5, p=2,
                     weights='uniform')

In [ ]:
# 6.4 Testando o modelo com Oversample

y_predito_over = modelo_over.predict(X_teste)

In [ ]:
# 6.5 Avaliando modelo com Oversample

print(classification_report(y_teste, y_predito_over))

              precision    recall  f1-score   support

       Fraud       0.20      0.91      0.32      2419
    No Fraud       1.00      1.00      1.00   1906367

    accuracy                           1.00   1908786
   macro avg       0.60      0.95      0.66   1908786
weighted avg       1.00      1.00      1.00   1908786



In [ ]:
# 6.6 Undersampling: diminuindo a classe majoritária

under_sampler = RandomUnderSampler(sampling_strategy = 'auto', random_state = 0)
X_treino_under, y_treino_under = under_sampler.fit_resample(X_treino, y_treino)
print(X_treino_under.shape)
print(y_treino_under.shape)

(11588, 6)
(11588,)


In [ ]:
# 6.7 Treinando modelo com os dados após Oversample

## 6.7.1 Instânciando modelo
modelo_under = KNeighborsClassifier(algorithm = 'auto',
                                    leaf_size = 30,
                                    metric = 'minkowski',
                                    metric_params = None,
                                    n_jobs = -1,
                                    n_neighbors = 5,
                                    p = 2,
                                    weights = 'uniform')

## 6.7.2 Treinando modelo
modelo_under.fit(X_treino_under, y_treino_under)

KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
                     metric_params=None, n_jobs=-1, n_neighbors=5, p=2,
                     weights='uniform')

In [ ]:
# 6.8 Testando o modelo com Undersampling

y_predito_under = modelo_under.predict(X_teste)

In [ ]:
# 6.9 Avaliando modelo com Undersampling

print(classification_report(y_teste, y_predito_under))

              precision    recall  f1-score   support

       Fraud       0.02      0.96      0.04      2419
    No Fraud       1.00      0.95      0.97   1906367

    accuracy                           0.95   1908786
   macro avg       0.51      0.96      0.51   1908786
weighted avg       1.00      0.95      0.97   1908786



**Observações:**

Após aplicação dos métodos Oversample e Undersample para balanceamento das classes o resultado apresentado mostra que não ouve melhora na classificação dos dados, pelo contrário, a classificação piorou em relação a classe Fraud.

Na sequência, aplicaremos o escalonamento/normalização aos dados e verificaremos se existe uma melhora nos resultados.

In [ ]:
# 6.10 Carregando biblioteca

from sklearn.preprocessing import StandardScaler

In [ ]:
# 6.11 Escalonando dados

## 6.11.1 Inicialização do escalonador
scaler = StandardScaler()

## 6.11.2 Ajustar o scaler nos dados de treino e aplicar a transformação
X_train_scaled = scaler.fit_transform(X_treino)

## 6.11.3 Aplicar a mesma transformação no conjunto de teste
X_test_scaled = scaler.transform(X_teste)

In [ ]:
# 6.12 Treinando modelo com os dados após Escalomanto

## 6.12.1 Instânciando modelo
modelo_scaler = KNeighborsClassifier(algorithm = 'auto',
                                    leaf_size = 30,
                                    metric = 'minkowski',
                                    metric_params = None,
                                    n_jobs = -1,
                                    n_neighbors = 5,
                                    p = 2,
                                    weights = 'uniform')

## 6.12.2 Treinando modelo
modelo_scaler.fit(X_train_scaled, y_treino)

KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
                     metric_params=None, n_jobs=-1, n_neighbors=5, p=2,
                     weights='uniform')

In [ ]:
# 6.13 Aplicando aos dados de teste

y_predito_scaler = modelo_scaler.predict(X_test_scaled)

In [ ]:
# 6.14 Avaliando resultado do modelo

print(classification_report(y_teste, y_predito_scaler))

              precision    recall  f1-score   support

       Fraud       0.92      0.74      0.82      2419
    No Fraud       1.00      1.00      1.00   1906367

    accuracy                           1.00   1908786
   macro avg       0.96      0.87      0.91   1908786
weighted avg       1.00      1.00      1.00   1908786



## **7. Considerações Finais**

<p align="justify">
O desempenho global do modelo foi impressionante, com acurácias de 100% nos primeiros dois resultados, o que inicialmente pode sugerir que o modelo está funcionando de maneira excelente. No entanto, a análise detalhada das métricas de precisão e recall revela um panorama mais complexo e valioso para ajustes futuros.

<p align="justify">
Desbalanceamento de Classes: O principal desafio identificado é o
desbalanceamento das classes, com uma clara predominância de transações legítimas (classe "No Fraud") em relação às transações fraudulentas (classe "Fraud"). Embora a acurácia global seja alta, o modelo tem uma baixa taxa de recall na classe "Fraud", especialmente nas primeiras iterações, indicando que o modelo falha em identificar um número significativo de fraudes, resultando em falsos negativos. A precisão da classe "Fraud" também foi relativamente baixa, refletindo o impacto do desbalanceamento nas previsões.

<p align="justify">
Impacto das Técnicas de Balanceamento: A aplicação das técnicas de Oversample e Undersample para balanceamento das classes não trouxe os resultados esperados. Em vez de melhorar a detecção de fraudes, essas abordagens levaram a uma redução na capacidade do modelo de identificar fraudes, com uma queda no f1-score da classe "Fraud". Isso sugere que essas técnicas podem não ter sido eficazes para esse caso específico, talvez devido a características dos dados ou a uma interação inesperada entre as técnicas e o modelo utilizado.

<p align="justify">
Importância da Normalização e Escalonamento: A normalização e o escalonamento dos dados foram identificados como etapas críticas para a melhoria do desempenho. A próxima fase do projeto deve focar em aplicar essas técnicas e avaliar seu impacto na precisão e recall da classe "Fraud", com a expectativa de que o ajuste dos dados possa ajudar a otimizar o modelo, tornando-o mais sensível a transações fraudulentas sem prejudicar a detecção de transações legítimas.

<p align="justify">
Avaliação da Classe Minoritária (Fraud): Embora o modelo tenha apresentado um desempenho perfeito na classe "No Fraud", a classe "Fraud" continua sendo o principal ponto de atenção. O baixo recall e f1-score indicam que há uma oportunidade significativa de melhorar a sensibilidade do modelo para detectar fraudes reais. Isso é fundamental para um sistema de detecção de fraudes em transações financeiras, onde o custo de falsos negativos (fraudes não detectadas) pode ser alto.

<p align="justify">
Em suma, embora os resultados iniciais sejam promissores, o projeto ainda enfrenta desafios significativos relacionados ao desbalanceamento de classes e à detecção eficiente de fraudes. Com as melhorias planejadas e ajustes nos parâmetros do modelo, é possível alcançar um sistema de detecção mais robusto e eficaz, que balanceie de maneira mais justa a precisão entre as classes, principalmente a classe minoritária "Fraud".